# Getting Started with Qdrant

Vector databases shine in many applications like [semantic search](https://en.wikipedia.org/wiki/Semantic_search) and [recommendation systems](https://en.wikipedia.org/wiki/Recommender_system), and in this tutorial, you will learn how to get started building such systems with one of the most popular and fastest growing vector databases in the market, [Qdrant](qdrant.tech).

## Table of contents

1. [Learning Outcomes](##-1.-Learning-Outcomes)
2. [Installation](##-2.-Installation)
3. [Getting Started](##-3.-Getting-Started)
    - [Adding Points](###-3.1-Adding-Points)
    - [Payload](###-3.2-Payloads)
    - [Search](###-3.3-Search)
4. [Recommendation System](##-4.-Recommendation-systems)
5. [Conclusion](##-5.-Conclusion) 

## 1. Learning outcomes

By the end of this tutorial, you will be able to:
- Create, update, and query collections of vectors using Qdrant.
- Conduct semantic search based on new data.
- Develop an intuition for the mechanics behind the recommendation API of Qdrant.
- Understand and get creative with the kind of data you can add to your payload.

## 2. Installation

The open source version of Qdrant is available as a Docker image. You can download the image and run it from any machine with Docker installed on it. If you don't have Docker installed, follow the instructions [here](https://docs.docker.com/get-docker/). After you have installed Docker, Terminal and download the Qdrant image:

```sh
docker pull qdrant/qdrant
```

Next, initialize Qdrant:

```sh
docker run -p 6333:6333 \
    -v $(pwd)/qdrant_storage:/qdrant/storage \
    qdrant/qdrant
```

You should see something like this:


<img src="https://github.com/qdrant/examples/blob/master/qdrant_101_getting_started/img/docker_qdrant_28_10_2023.png?raw=1" width="50%">



In [1]:
%pip install qdrant-client pandas numpy faker

Note: you may need to restart the kernel to use updated packages.


## 3. Getting started

The two modules we are going to use are `QdrantClient` and `models`. The former lets you connect to Qdrant or to run an in-memory database by switching the parameter `location=` to `":memory:"`. The latter gives you access to most functionalities you need to interact with Qdrant.

In [2]:
from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import CollectionStatus

We'll start by instantiating our client using `host="localhost"` and `port=6333` (as it is the default port we used earlier with Docker). You can also follow along with the `location=":memory:"` option commented out below.

In [3]:
client = QdrantClient(host="localhost", port=6333)
client

In [4]:
# client = QdrantClient(location=":memory:")
# client

---
**Note:** In OLTP and OLAP databases we call specific bundles of rows and columns **Tables**. However, in vector databases, the rows are known as **Vectors**, while the columns are **Dimensions**. The combination of the two (plus some metadata) is a [**Collection**](https://qdrant.tech/documentation/concepts/collections/).

Just as we can create many tables in an OLTP or an OLAP database, we can create many collections in a vector database like Qdrant using one of its clients. The key difference to note is that when we create a collection in Qdrant, we need to specify the width of the collection (i.e. the length of the vector or amount of dimensions) beforehand with the parameter `size=...`, as well as the distance metric with the parameter `distance=...`.

The distances currently supported by Qdrant are [**Cosine Similarity**](https://en.wikipedia.org/wiki/Cosine_similarity), [**Dot Product**](https://en.wikipedia.org/wiki/Dot_product), and [**Euclidean Distance**](https://en.wikipedia.org/wiki/Euclidean_distance).

---

Let's create our first collection and have the vectors be of size 100 with a distance set to **Cosine Similarity**.

In [5]:
my_collection = "first_collection"
 
 
if client.collection_exists(my_collection):
    client.delete_collection(my_collection)
    print(f"Coleção '{my_collection}' removida.")
 
first_collection = client.create_collection(
    collection_name=my_collection,
    vectors_config=models.VectorParams(size=100,distance=models.Distance.COSINE)
)
print(first_collection)

True


We can extract information related to the health of our collection by retrieving the collection with our client. In addition, we can use this information for testing purposes, which can be very beneficial while in development mode.

In [6]:
collection_info = client.get_collection(collection_name=my_collection)
list(collection_info)

[('status', <CollectionStatus.GREEN: 'green'>),
 ('optimizer_status', <OptimizersStatusOneOf.OK: 'ok'>),
 ('warnings', None),
 ('indexed_vectors_count', 0),
 ('points_count', 0),
 ('segments_count', 4),
 ('config',
  CollectionConfig(params=CollectionParams(vectors=VectorParams(size=100, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optimization_threads=None), wal_config=WalConfig(wal_

Two important takeaways:

1. When you initiated the Docker image, you created a local directory called, `qdrant_storage`. This is where all of your collections and their metadata will be stored.

    Qdrant can use one of two options for [storage](https://qdrant.tech/documentation/concepts/storage/):
    - **in-memory** storage, which stores all vectors in RAM and has the highest speed since disk access is required only for persistence)
    - **memmap** storage, which creates a virtual address space associated with the file on disk. You can have a look at that directory in a *nix system with `tree qdrant_storage -L 2`, and something similar to the following output should come up for you.

        ```bash
        qdrant_storage
        ├── aliases
        │   └── data.json
        ├── collections
        │   └── my_first_collection
        └── raft_state


### 3.1 Adding points

[Points](https://qdrant.tech/documentation/concepts/points/) are a central entity that Qdrant operates with. They contain records consisting of a vector, an optional `id`, and an optional `payload`.

The optional id can be represented by [unsigned integers](https://en.wikipedia.org/wiki/Integer_(computer_science)) or [UUID](https://en.wikipedia.org/wiki/Universally_unique_identifier)s. For this tutorial, we will use a straightforward range of numbers.

You can use [NumPy](https://numpy.org/) to create a matrix of dummy data containing 1,000 vectors and 100 dimensions.  Then, you can represent the values as `float64` numbers between -1 and 1. For simplicity, imagine that each of these vectors represents one of our favorite songs. Then, each column would represent a unique characteristic of the song; e.g. the tempo, the beats, the pitch of the voice(s) of the singer(s).

In [7]:
import numpy as np
data = np.random.uniform(low=-1.0, high=1.0, size=(1_000, 100))
type(data[0, 0]), data[:2, :20]

(numpy.float64,
 array([[-0.26372568, -0.15081956,  0.08628753, -0.93846593, -0.16878617,
          0.34074502, -0.36229223, -0.42647628, -0.76132699, -0.92912647,
          0.98033259,  0.85905212,  0.95917297,  0.11213236,  0.50287616,
         -0.55582791,  0.25271359, -0.02324697,  0.55727277,  0.35036868],
        [ 0.53175481,  0.58553623, -0.64862476,  0.36651354,  0.03063883,
         -0.83939722, -0.4714324 ,  0.47795451, -0.96094839,  0.32021003,
          0.31994975,  0.56710307, -0.89396441,  0.60692198,  0.83785166,
          0.07834162, -0.38972682,  0.67606127,  0.81322616, -0.71323519]]))

Now you can create an index for your vectors.

In [8]:
index = list(range(len(data)))
index[-10:]

[990, 991, 992, 993, 994, 995, 996, 997, 998, 999]

Once a collection has been created, you can fill it in with `client.upsert()`. You need to provide the collection's name and the appropriate uploading process from our `models` module, in this case, [`Batch`](https://qdrant.tech/documentation/points/#upload-points).

**Note:** Qdrant can only take in native Python iterables like lists and tuples. This is why you will notice the `.tolist()` method attached to our `data` matrix below.

In [9]:
client.upsert(
    collection_name=my_collection,
    points=models.Batch(
        ids=index,
        vectors=data.tolist()
    )
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

You can retrieve specific points based on their ID (for example, song X with ID 100) and get some additional information from that result.

In [10]:
client.retrieve(
    collection_name=my_collection,
    ids=[100],
    with_vectors=True # the default is False
)

[Record(id=100, payload={}, vector=[-0.0739143, 0.062745914, 0.07307422, -0.096073054, 0.13127932, -0.12978359, 0.085935816, -0.07211814, -0.10143877, -0.15798654, 0.0421215, 0.033987854, 0.038952466, 0.16536766, 0.13393362, -0.07852628, 0.052172437, 0.13197249, 0.12711841, 0.10849627, -0.10709053, -0.16103224, -0.02274866, 0.036041234, -0.09591749, -0.11900258, -0.10302162, 0.14788459, 0.122063436, 0.07616956, 0.017259747, -0.15478717, 0.13266318, -0.06525492, -0.038122423, -0.019570505, -0.049460214, 0.14151703, -0.11881361, 0.12440789, -0.0137349265, 0.07598499, -0.03926576, -0.15665245, 0.04625941, 0.0929224, -0.124756426, -0.12661244, -0.074957624, 0.014753411, 0.06450542, 0.02239464, 0.13665505, -0.16414419, 0.15358557, -0.058316384, -0.02822659, 0.055773396, -0.054607153, -0.121963285, -0.14379789, -0.034166064, 0.015297111, 0.09986658, 0.10179503, -0.017800504, 0.08029868, 0.025115065, 0.12232232, -0.029031152, -0.14291717, 0.046886593, -0.16522422, 0.1240265, -0.07706088, 0.03

You can also update the collection one point at a time; e.g. as new data is coming in.

In [11]:
def create_song():
    return np.random.uniform(low=-1.0, high=1.0, size=100).tolist()

In [12]:
client.upsert(
    collection_name=my_collection,
    points=[
        models.PointStruct(
            id=1000,
            vector=create_song(),
        )
    ]
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

We can also delete a point in a straightforward fashion.

In [13]:
# this will show the amount of vectors BEFORE deleting the one we just created
client.count(
    collection_name=my_collection,
    exact=True,
)

CountResult(count=1001)

In [14]:
client.delete(
    collection_name=my_collection,
    points_selector=models.PointIdsList(
        points=[1000],
    ),
)

UpdateResult(operation_id=3, status=<UpdateStatus.COMPLETED: 'completed'>)

In [15]:
# this will show the amount of vectors AFTER deleting them
client.count(
    collection_name=my_collection,
    exact=True,
)

CountResult(count=1000)

### 3.2 Payloads

With Qdrant you can store additional information alongside vectors. This is called a [payload](https://qdrant.tech/documentation/payload/) and it is represented as JSON objects. With these payloads, not only can you retrieve information when you search the database, but you can also filter your search by the parameters in the payload, and we'll see how in a second.

Following the narrative that our dummy vectors "represent a song," in a semantic search system for this kind of data, you would want to retrieve the song file, its URL, the artist or the genre, among others.

We will take advantage of a Python package called `faker` and create a bit of fake information to add to our payload to test this functionality.

For each vector, you can create list of dictionaries containing the artist's name, the song, a url to the song, the year in which it was released, and the country where it originated from.

In [16]:
from faker import Faker
fake_something = Faker()

payload = []

for i in range(len(data)):
    payload.append(
        {
            "artist":   fake_something.name(),
            "song":     " ".join(fake_something.words()),
            "url_song": fake_something.url(),
            "year":     fake_something.year(),
            "country":  fake_something.country()
        }
    )

payload[:3]

[{'artist': 'Brandy Green',
  'song': 'move read painting',
  'url_song': 'https://www.mays-stephens.net/',
  'year': '1974',
  'country': 'Austria'},
 {'artist': 'Eric Terrell',
  'song': 'already specific reality',
  'url_song': 'https://andrade-marks.biz/',
  'year': '1979',
  'country': 'Suriname'},
 {'artist': 'Cody Price',
  'song': 'standard collection environment',
  'url_song': 'http://meyer.com/',
  'year': '2022',
  'country': 'Jersey'}]

You can upsert your Points (ids, data, and payload), with the same `client.upsert()` method you used earlier.

In [17]:
client.upsert(
    collection_name=my_collection,
    points=models.Batch(
        ids=index,
        vectors=data.tolist(),
        payloads=payload
    )
)

UpdateResult(operation_id=4, status=<UpdateStatus.COMPLETED: 'completed'>)

If you want to retrieve this info, use the `client.retrieve()` method.

In [18]:
resutls = client.retrieve(
    collection_name=my_collection,
    ids=[10, 50, 100, 500],
    with_vectors=False
)

type(resutls), resutls

(list,
 [Record(id=10, payload={'artist': 'Maureen Guerrero', 'song': 'stand way listen', 'url_song': 'http://campbell.com/', 'year': '1992', 'country': "Lao People's Democratic Republic"}, vector=None, shard_key=None, order_value=None),
  Record(id=50, payload={'artist': 'Yvonne Calhoun', 'song': 'local until and', 'url_song': 'http://www.berry-campbell.info/', 'year': '1990', 'country': 'Tunisia'}, vector=None, shard_key=None, order_value=None),
  Record(id=100, payload={'artist': 'Leah Kelly', 'song': 'lawyer lawyer billion', 'url_song': 'https://eaton.net/', 'year': '1972', 'country': 'Pitcairn Islands'}, vector=None, shard_key=None, order_value=None),
  Record(id=500, payload={'artist': 'Robert Zimmerman', 'song': 'something somebody number', 'url_song': 'http://www.lambert-parker.info/', 'year': '2018', 'country': 'Central African Republic'}, vector=None, shard_key=None, order_value=None)])

The response is a list with records and each element inside a record can be accessed as an attribute, e.g. `.payload` or `.id`.

In [19]:
resutls[0].payload

{'artist': 'Maureen Guerrero',
 'song': 'stand way listen',
 'url_song': 'http://campbell.com/',
 'year': '1992',
 'country': "Lao People's Democratic Republic"}

In [20]:
resutls[0].id

10

### 3.3 Search

Now that you have your vectors with an ID and a payload, you can start searching for content when new music gets selected.

Assume that a new song (like ["living la vida loca"](https://www.youtube.com/watch?v=p47fEXGabaY&ab_channel=RickyMartinVEVO) by Ricky Martin) comes in and our model immediately transforms it into a vector. Since we don't want a large amount of values back, let's limit the search to a few points.

In [21]:
living_la_vida_loca = create_song()

In [22]:
client.query_points(
    collection_name=my_collection,
    query=living_la_vida_loca,
    limit=3
).points

[ScoredPoint(id=757, version=4, score=0.31476814, payload={'artist': 'Jonathan Dalton', 'song': 'run late approach', 'url_song': 'https://meyers.com/', 'year': '1972', 'country': 'Romania'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=731, version=4, score=0.28766927, payload={'artist': 'Ian Simmons', 'song': 'summer environment traditional', 'url_song': 'https://phillips.com/', 'year': '1983', 'country': 'Isle of Man'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=79, version=4, score=0.27206677, payload={'artist': 'David Hart', 'song': 'already raise between', 'url_song': 'http://elliott.com/', 'year': '2014', 'country': 'Haiti'}, vector=None, shard_key=None, order_value=None)]

Assume you only want Australian songs recommended to you. For this, you can filter the query using the information in the payload. You have to first create a filter object and then pass it to the search method as an argument to the parameter `query_filter=`.

In [23]:
aussie_songs = models.Filter(
    must=[models.FieldCondition(key="country", match=models.MatchValue(value="Australia"))]
)
type(aussie_songs)

qdrant_client.http.models.models.Filter

In [24]:
client.query_points(
    collection_name=my_collection,
    query=living_la_vida_loca,
    query_filter=aussie_songs,
    limit=2
).points

[ScoredPoint(id=203, version=4, score=0.11338389, payload={'artist': 'Jason Morgan MD', 'song': 'guess southern away', 'url_song': 'https://bell.biz/', 'year': '1987', 'country': 'Australia'}, vector=None, shard_key=None, order_value=None)]

Lastly, assume we want aussie songs but we don't care how new or old these songs are. Exclude the year from the payload.

In [25]:
client.query_points(
    collection_name=my_collection,
    query=living_la_vida_loca,
    query_filter=aussie_songs,
    with_payload=models.PayloadSelectorExclude(exclude=["year"]),
    limit=5
).points

[ScoredPoint(id=203, version=4, score=0.11338389, payload={'artist': 'Jason Morgan MD', 'song': 'guess southern away', 'url_song': 'https://bell.biz/', 'country': 'Australia'}, vector=None, shard_key=None, order_value=None)]

## 4. Recommendation systems

A recommendation system is a technology that suggests items or content to users based on their preferences, interests, or past behavior. In its most widely-used form, recommendation systems work by analyzing data about you and other users. The system looks at your previous choices, such as movies you've watched, products you've bought, or articles you've read. It then compares this information with data from other people who have similar tastes or interests.

Such systems are used in various companies such as Netflix, Amazon, Tik-Tok, and Spotify. They aim to personalize your experience, save you time searching for things you might like, or introduce you to new and relevant content that you may not have discovered otherwise.

Qdrant's API supports such a system, letting you account for user feedback. For example, you can recommend songs based on user likes (👍) or exclude similar ones to content users have disliked (👎).

To do this, use the `RecommendQuery` method and consider the following elements:

- `collection_name=` - the collection from which the vectors are selected
- `query_filter=` - optional filter to apply to your search
- `negative=` - optionally, specify the `id` of disliked songs to exclude other semantically similar songs
- `positive=` - in case of liked songs, specify their `id` to exclude similar songs (required)
- `limit=` - specifies how many songs to show to the user

Imagine there are two songs, "[Suegra](https://www.youtube.com/watch?v=p7ff5EntWsE&ab_channel=RomeoSantosVEVO)" by Romeo Santos and "[Worst Behavior](https://www.youtube.com/watch?v=U5pzmGX8Ztg&ab_channel=DrakeVEVO)" by Drake represented by the ids 17 and 120 respectively. Let's see what we would get with the former being a 👍 and the latter being a 👎.

Notice that, while the similarity scores are completely random for this example, it is important to we pay attention to the scores retrieved when serving recommendations in production. Even if you get 5 vectors back, it might be more useful to show random results, rather than vectors that are 0.012 similar to the query vector. With this in mind, you can actually set a threshold for our vectors with the `score_threshold=` parameter.

In [26]:
client.query_points(
    collection_name=my_collection,
    query=models.RecommendQuery(recommend=models.RecommendInput(
        positive=[17],
        negative=[120, 180]
    )),
    score_threshold=0.22,
    limit=5
).points

[ScoredPoint(id=188, version=4, score=0.3639396, payload={'artist': 'Caitlin Jackson', 'song': 'south his chance', 'url_song': 'http://hill.com/', 'year': '1984', 'country': 'Dominica'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=613, version=4, score=0.3286008, payload={'artist': 'Zachary Barnett', 'song': 'impact trouble meeting', 'url_song': 'http://www.stone.com/', 'year': '1970', 'country': 'Taiwan'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=75, version=4, score=0.3138233, payload={'artist': 'Lisa Murray', 'song': 'always by hundred', 'url_song': 'https://www.crane-lee.com/', 'year': '2016', 'country': 'Lebanon'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=580, version=4, score=0.29644215, payload={'artist': 'Lori Jones', 'song': 'PM age decade', 'url_song': 'http://chase-randall.info/', 'year': '1973', 'country': 'Antigua and Barbuda'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=839, version=4, 

Lastly, you can add filters in the same way as you did before. Note that these filters could be tags that your users get to pick such as, for example, genres including `reggeaton`, `bachata`, and `salsa` (sorry Drake), or the language of the song.

In [27]:
client.query_points(
    collection_name=my_collection,
    query=models.RecommendQuery(recommend=models.RecommendInput(
        positive=[17],
        negative=[120, 180]
    )),
    query_filter=models.Filter(
        must=[models.FieldCondition(key="country", match=models.MatchValue(value="Dominican Republic"))]
    ),
    limit=5
).points

[ScoredPoint(id=976, version=4, score=0.1981943, payload={'artist': 'Ryan Perkins', 'song': 'tax establish improve', 'url_song': 'http://www.stevens.info/', 'year': '1998', 'country': 'Dominican Republic'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=756, version=4, score=0.02811126, payload={'artist': 'Jeffrey Wright', 'song': 'month offer evidence', 'url_song': 'http://morrow.com/', 'year': '1986', 'country': 'Dominican Republic'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=586, version=4, score=-0.009532735, payload={'artist': 'Melanie Christensen', 'song': 'crime include practice', 'url_song': 'https://miller.biz/', 'year': '1975', 'country': 'Dominican Republic'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=195, version=4, score=-0.012149429, payload={'artist': 'Veronica Jones', 'song': 'million at goal', 'url_song': 'https://hall.biz/', 'year': '1983', 'country': 'Dominican Republic'}, vector=None, shard_key=None, order

## 5. Conclusion

To wrap up, we have explored a bit of the fascinating world of vector databases, and we learned that these databases provide efficient storage and retrieval of high-dimensional vectors, making them ideal for similarity-based search tasks and recommendation systems. Both of these use cases can be applied in a variety of industries while helping us unlock new levels of information retrieval. In particular, recommendation systems built with Qdrant provide developers with enough flexibility to add and subtract data points that users liked or disliked, respectively, and even set up a threshold for how similar a recommendation must be before our applications can serve it.
